In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
q_startup_current = spark.sql("""
WITH startup_events AS (
    SELECT
        timestamp,
        Motor_current,
        COMP,
        LAG(COMP,1,1) OVER (ORDER BY timestamp) AS prev_comp
    FROM sensor
)
SELECT
    ROUND(AVG(Motor_current),4) AS dong_dien_khoi_dong_tb,
    ROUND(MIN(Motor_current),4) AS dong_dien_khoi_dong_min,
    ROUND(MAX(Motor_current),4) AS dong_dien_khoi_dong_max,
    COUNT(*) AS so_lan_khoi_dong
FROM startup_events
WHERE prev_comp = 1
  AND COMP = 0
""")

q_startup_current.show()

In [ ]:
# Q7: PHÁT HIỆN DÒNG ĐIỆN KHỞI ĐỘNG BẤT THƯỜNG
q7 = spark.sql('''
    SELECT *
    FROM (
        SELECT
            timestamp, date, hour, COMP,
            Motor_current AS dong_dien_luc_khoi_dong,
            LAG(COMP, 1, 1) OVER (ORDER BY timestamp) AS prev_comp,
            LEAD(Motor_current, 1) OVER (ORDER BY timestamp) AS dong_dien_sau_10s,
            LEAD(Motor_current, 2) OVER (ORDER BY timestamp) AS dong_dien_sau_20s,
            LEAD(Motor_current, 3) OVER (ORDER BY timestamp) AS dong_dien_sau_30s,
            GREATEST(
                Motor_current,
                LEAD(Motor_current, 1) OVER (ORDER BY timestamp),
                LEAD(Motor_current, 2) OVER (ORDER BY timestamp),
                LEAD(Motor_current, 3) OVER (ORDER BY timestamp)
            ) AS dong_dien_dinh
        FROM sensor)
    WHERE dong_dien_luc_khoi_dong > 0
      AND prev_comp = 1
      AND COMP = 0
    ORDER BY dong_dien_dinh DESC
    LIMIT 15''')
print("\n========== EXECUTION PLAN ==========\n")
q7.explain(True)
df_q7 = q7.toPandas()
print("--- TOP 15 LẦN KHỞI ĐỘNG CÓ DÒNG ĐIỆN ĐỈNH CAO NHẤT ---")
try:    display(df_q7)
except NameError:    print(df_q7.to_string(index=False))